# Risk Parity

Risk parity allocates capital so assets contribute more evenly to portfolio risk.

Abbreviations used in this notebook:

- **IVW**: Inverse Volatility Weighting.
- **RC**: Risk Contribution.
- **MRC**: Marginal Risk Contribution.
- **SD**: Standard Deviation.
- **IR**: Information Ratio.

## 1. Intuition

Equal-weight portfolios allocate the same capital to each asset, but high-volatility assets can dominate risk. Risk parity shifts more capital to lower-volatility assets and less to higher-volatility assets.

## 2. Mathematics

**Portfolio volatility:**

$$
\sigma_p = \sqrt{w^T\Sigma w}
$$

Where:

- $w$ = vector of portfolio weights
- $\Sigma$ = covariance matrix of asset returns
- $\sigma_p$ = portfolio volatility

**Marginal risk contribution:**

$$
MRC_i = \frac{(\Sigma w)_i}{\sigma_p}
$$

Where:

- $w$ = vector of portfolio weights
- $\Sigma$ = covariance matrix of asset returns
- $\sigma_p$ = portfolio volatility
- $MRC_i$ = marginal risk contribution of asset $i$
- $RC_i$ = total risk contribution of asset $i$

**Total risk contribution:**

$$
RC_i = w_i \times MRC_i
$$

Where:

- $w_i$ = portfolio weight assigned to asset $i$
- $MRC_i$ = marginal risk contribution of asset $i$
- $RC_i$ = total risk contribution of asset $i$

**Inverse volatility weight:**

$$
w_i = \frac{1/\sigma_i}{\sum_j 1/\sigma_j}
$$

Where:

- $w_i$ = portfolio weight assigned to asset $i$
- $\sigma_i$ = volatility of asset $i$

## 3. Implementation

We compare equal weights with inverse-volatility weights across the synthetic universe.

In [ ]:
import importlib.util
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "GUIDELINES.md").exists())
helper_path = project_root / "05_strategies" / "strategy_utils.py"
spec = importlib.util.spec_from_file_location("strategy_utils", helper_path)
strategy_utils = importlib.util.module_from_spec(spec)
spec.loader.exec_module(strategy_utils)

plt.style.use("seaborn-v0_8-whitegrid")
prices, returns, fundamentals = strategy_utils.generate_strategy_universe()
benchmark = returns.mean(axis=1)

asset_returns = returns.iloc[:, :8]
equal_weights = pd.Series(1 / asset_returns.shape[1], index=asset_returns.columns)
ivw_weights = strategy_utils.inverse_volatility_weights(asset_returns)

equal_returns = asset_returns @ equal_weights
ivw_returns = asset_returns @ ivw_weights

weights = pd.DataFrame({"equal_weight": equal_weights, "inverse_volatility": ivw_weights})
weights

In [ ]:
cov = asset_returns.cov() * 252

def risk_contribution(weight_series):
    w = weight_series.to_numpy()
    port_vol = np.sqrt(w.T @ cov.to_numpy() @ w)
    mrc = cov.to_numpy() @ w / port_vol
    rc = w * mrc
    return pd.Series(rc / rc.sum(), index=weight_series.index)

risk_contributions = pd.DataFrame({
    "equal_weight": risk_contribution(equal_weights),
    "inverse_volatility": risk_contribution(ivw_weights),
})

summary = pd.DataFrame({
    "equal_weight": strategy_utils.performance_summary(equal_returns),
    "inverse_volatility": strategy_utils.performance_summary(ivw_returns),
})
summary.round(4)

## 4. Visualization

Risk parity analysis should compare capital weights with risk contributions.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
weights.plot(kind="bar", ax=axes[0], color=["#9a6b2f", "#2f6f8f"])
axes[0].set_title("Capital Weights")
axes[0].set_ylabel("Weight")
axes[0].tick_params(axis="x", rotation=35)
axes[0].yaxis.set_major_formatter(lambda x, pos: f"{x:.0%}")

risk_contributions.plot(kind="bar", ax=axes[1], color=["#9a6b2f", "#2f6f8f"])
axes[1].set_title("Risk Contributions")
axes[1].set_ylabel("Share of risk")
axes[1].tick_params(axis="x", rotation=35)
axes[1].yaxis.set_major_formatter(lambda x, pos: f"{x:.0%}")
plt.tight_layout(); plt.show()

In [ ]:
(1 + pd.DataFrame({"Equal Weight": equal_returns, "Inverse Volatility": ivw_returns})).cumprod().plot(figsize=(10, 4), color=["#9a6b2f", "#2f6f8f"])
plt.title("Risk Parity-Style Portfolio vs Equal Weight")
plt.ylabel("Growth of 1")
plt.tight_layout(); plt.show()

## 5. Application

Risk parity is often used across asset classes, not single stocks. It can reduce risk concentration, but it may allocate heavily to low-volatility assets and may use leverage in institutional settings.

In [ ]:
pd.concat({"weights": weights, "risk_contribution": risk_contributions}, axis=1).round(3)

## 6. Reflection

- Equal capital weight does not mean equal risk weight.
- Inverse volatility is a simple approximation of risk parity.
- Correlations also affect risk contributions.
- Risk parity can underperform when low-volatility assets sell off together.

Questions to answer after running the notebook:

1. Which assets dominate risk in the equal-weight portfolio?
2. Did inverse-volatility weighting reduce volatility?
3. Why does correlation matter for risk contribution?
4. How would leverage change a risk parity portfolio?